<a href="https://colab.research.google.com/github/samrat-dhakal-11/Google-Colab-Notebooks/blob/main/Middlewares_n_Guardials.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai langgraph langchain-core langchain-tavily pydantic
!pip install -U "psycopg[binary]"
!pip install -U pgvector langchain-postgres
print("Installation Complete")

In [2]:
import os
from langchain_openai import ChatOpenAI
from google.colab import userdata
DB_URL=userdata.get("NEON_URL")
os.environ['OPENAI_API_KEY']=userdata.get("OPENAI_API_KEY")
llm=ChatOpenAI(model="gpt-5.4-mini")

Guardrails

In [8]:
ban_words=["kill","hack","porn","sex","system prompt","ignore","ignore all"]
user="How to kill a person"
def dm(msg):
  for keys in ban_words:
    if keys in msg:
      return "Blocked: Please enter a freindly message"
  return "Sure ......"
dm(user)

'Blocked: Please enter a freindly message'

Regex

In [11]:
import re
from typing import TypedDict
ban_pattern=re.compile(r"\b("+"|".join(map(re.escape,ban_words))+r")\b",re.IGNORECASE)

In [17]:
from typing import TypedDict
class State(TypedDict):
    message: str
    reply: str

def guard(state:State):
  msg=state["message"]
  sr=ban_pattern.search(msg)
  if sr:
    return {'reply':f"Blocked word:  {sr.group()}"}
  return {'reply':llm.invoke(msg).content}

print(guard({"message": "How to kill a person", "reply": ""}),"\n\n")
print(guard({"message": "Whos is albert einsten", "reply": ""}))

{'reply': 'Blocked word:  kill'} 


{'reply': 'Albert Einstein was a famous physicist, best known for developing the theory of relativity, including the equation **E = mc²**. He made major contributions to modern physics and won the **Nobel Prize in Physics in 1921** for his explanation of the photoelectric effect.\n\nIf you want, I can also give you:\n- a very short version,\n- a kid-friendly explanation, or\n- a summary of his biggest discoveries.'}


In [19]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    message: str
    reply: str
    blocked: bool

def guard(state: State):
    sr = ban_pattern.search(state["message"])
    if sr:
        return {"blocked": True, "reply": f"Blocked word: {sr.group()}"}
    return {"blocked": False}

def route(state: State):
    # Returns the name of the next node
    return "refuse" if state["blocked"] else "model"

def refuse(state: State):
    # reply was already set by the guard, so nothing to change here
    return {}

def model(state: State):
    return {"reply": llm.invoke(state["message"]).content}

builder = StateGraph(State)
builder.add_node("guard", guard)
builder.add_node("refuse", refuse)
builder.add_node("model", model)

builder.add_edge(START, "guard")
builder.add_conditional_edges("guard", route, {"refuse": "refuse", "model": "model"})
builder.add_edge("refuse", END)
builder.add_edge("model", END)

graph = builder.compile()

print(graph.invoke({"message": "How to kill a person", "reply": "", "blocked": False})["reply"])
print(graph.invoke({"message": "Who is Albert Einstein", "reply": "", "blocked": False})["reply"])

Blocked word: kill
Albert Einstein was a theoretical physicist widely regarded as one of the most influential scientists in history. He is best known for:

- **The theory of relativity**, including the famous equation **E = mc²**
- **Explaining the photoelectric effect**, which helped establish quantum theory
- Major contributions to **modern physics** and our understanding of space, time, and gravity

He was born in **1879** in Germany and died in **1955** in the United States. Beyond science, Einstein was also known for his humanitarian views and outspoken opinions on politics and peace.


Loggers

In [32]:

import logging
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("guard")

class State(TypedDict):
    message: str
    reply: str
    blocked: bool
    audit: Annotated[list, operator.add]  # nodes append to this list

def guard(state: State):
    sr = ban_pattern.search(state["message"])
    if sr:
        return {"blocked": True, "reply": f"Blocked word: {sr.group()}"}
    return {"blocked": False}

def route(state: State):
    return "refuse" if state["blocked"] else "model"

def refuse(state: State):
    return {}

def model(state: State):
    return {"reply": llm.invoke(state["message"]).content}

def log_event(state: State):
    entry = {"message": state["message"], "blocked": state["blocked"]}
    if state["blocked"]:
        logger.warning("BLOCKED: %r reason=%s", state["message"], state["reply"])
    else:
        logger.info("ALLOWED: %r", state["message"])
    return {"audit": [entry]}  # appended, not replaced

builder = StateGraph(State)
builder.add_node("guard", guard)
builder.add_node("refuse", refuse)
builder.add_node("model", model)
builder.add_node("log_event", log_event)

builder.add_edge(START, "guard")
builder.add_conditional_edges("guard", route, {"refuse": "refuse", "model": "model"})
builder.add_edge("refuse", "log_event")
builder.add_edge("model", "log_event")
builder.add_edge("log_event", END)

graph = builder.compile()

for msg in ["How to kill a person", "Who is Albert Einstein"]:
    result = graph.invoke({"message": msg, "reply": "", "blocked": False, "audit": []})
    print("REPLY:", result["reply"])
    print("AUDIT:", result["audit"], "\n")

guard -> WARNING : BLOCKED: 'How to kill a person' reason=Blocked word: kill


REPLY: Blocked word: kill
AUDIT: [{'message': 'How to kill a person', 'blocked': True}] 



httpx -> INFO : HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
guard -> INFO : ALLOWED: 'Who is Albert Einstein'


REPLY: Albert Einstein was a theoretical physicist best known for developing the theory of relativity, including the famous equation **E = mc²**. He is considered one of the most influential scientists in history.

A few key facts:
- Born in **1879** in Germany
- Developed **special relativity** and later **general relativity**
- Won the **1921 Nobel Prize in Physics** for explaining the **photoelectric effect**
- His work helped shape modern physics, including our understanding of space, time, gravity, and energy

If you want, I can also give you a **short biography**, **his major discoveries**, or a **simple explanation of relativity**.
AUDIT: [{'message': 'Who is Albert Einstein', 'blocked': False}] 



Adding Evaluator Node

In [34]:

import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    message: str
    reply: str
    blocked: bool
    attempts: int
    verdict: str
    audit: Annotated[list, operator.add]

def guard(state: State):
    sr = ban_pattern.search(state["message"])
    if sr:
        return {"blocked": True, "reply": f"Blocked word: {sr.group()}"}
    return {"blocked": False}

def route_safety(state: State):
    return "refuse" if state["blocked"] else "model"

def refuse(state: State):
    return {}

def model(state: State):
    # Add a stricter instruction on retries
    prompt = state["message"]
    if state["attempts"] > 0:
        prompt = "Answer briefly and politely, with no banned or unsafe content.\n\n" + prompt
    return {"reply": llm.invoke(prompt).content, "attempts": state["attempts"] + 1}

def evaluate(state: State):
    # Output guard: reuse the same ban pattern on the reply
    if ban_pattern.search(state["reply"]):
        return {"verdict": "fail"}
    return {"verdict": "pass"}

def route_verdict(state: State):
    if state["verdict"] == "pass":
        return "log_event"
    if state["attempts"] >= 2:  # give up after 2 tries
        return "refuse_output"
    return "model"  # retry

def refuse_output(state: State):
    return {"reply": "Sorry, I can't give a safe answer to that."}

def log_event(state: State):
    entry = {"message": state["message"], "blocked": state["blocked"], "verdict": state["verdict"]}
    return {"audit": [entry]}

builder = StateGraph(State)
builder.add_node("guard", guard)
builder.add_node("refuse", refuse)
builder.add_node("model", model)
builder.add_node("evaluate", evaluate)
builder.add_node("refuse_output", refuse_output)
builder.add_node("log_event", log_event)

builder.add_edge(START, "guard")
builder.add_conditional_edges("guard", route_safety, {"refuse": "refuse", "model": "model"})
builder.add_edge("refuse", "log_event")
builder.add_edge("model", "evaluate")
builder.add_conditional_edges("evaluate", route_verdict, {
    "log_event": "log_event",
    "refuse_output": "refuse_output",
    "model": "model",
})
builder.add_edge("refuse_output", "log_event")
builder.add_edge("log_event", END)

graph = builder.compile()

for msg in ["How to kill a person", "Who is Albert Einstein"]:
    result = graph.invoke({"message": msg, "reply": "", "blocked": False,
                           "attempts": 0, "verdict": "", "audit": []})
    print("REPLY:", result["reply"][:120], "\n")

REPLY: Blocked word: kill 



httpx -> INFO : HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


REPLY: Albert Einstein was a theoretical physicist best known for developing the theory of relativity, including the famous equ 



Postgress and Pg vector Setup

Pg Vector

In [44]:
import psycopg

with psycopg.connect(DB_URL) as conn:
    conn.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    conn.commit()

In [ ]:

import os
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


vectorstore=PGVector(
  embeddings=embeddings,
  collection_name="knowledge",
  connection=DB_URL,
  use_jsonb=True
)


docs = [
    Document(page_content="Euler's totient counts integers coprime to n.", metadata={"domain": "math"}),
    Document(page_content="Albert Einstein developed the theory of relativity.", metadata={"domain": "general"}),
]

vectorstore.add_documents(docs)

retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={'k':2})
retriever.invoke("integer")

Postgres

In [ ]:
!pip install -U langgraph-checkpoint-postgres

In [62]:

import os
from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()  # creates the checkpoint tables if they don't exist

    graph = builder.compile(checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "user-1"}}

    for msg in ["Who is Albert Einstein", "How to kill a person", "Who discovered penicillin"]:
        result = graph.invoke(
            {"message": msg, "reply": "", "blocked": False,
             "attempts": 0, "verdict": ""},
            config,
        )
        print("REPLY:", result["reply"][:80], "\n")

    final = graph.get_state(config)
    print("AUDIT:", final.values["audit"])

httpx -> INFO : HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


REPLY: Albert Einstein was a theoretical physicist born in 1879 in Germany, best known  

REPLY: Blocked word: kill 



httpx -> INFO : HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


REPLY: Penicillin was discovered by **Alexander Fleming** in **1928**. 

AUDIT: [{'message': 'Who is Albert Einstein', 'blocked': False, 'verdict': 'pass'}, {'message': 'How to kill a person', 'blocked': True, 'verdict': ''}, {'message': 'Who discovered penicillin', 'blocked': False, 'verdict': 'pass'}]


In [64]:

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    graph = builder.compile(checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "user-1"}}
    print( graph.get_state(config))

StateSnapshot(values={'message': 'Who discovered penicillin', 'reply': 'Penicillin was discovered by **Alexander Fleming** in **1928**.', 'blocked': False, 'attempts': 1, 'verdict': 'pass', 'audit': [{'message': 'Who is Albert Einstein', 'blocked': False, 'verdict': 'pass'}, {'message': 'How to kill a person', 'blocked': True, 'verdict': ''}, {'message': 'Who discovered penicillin', 'blocked': False, 'verdict': 'pass'}]}, next=(), config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dc4bd-c188-6d8b-800f-aafff34479a0'}}, metadata={'step': 15, 'source': 'loop', 'parents': {}}, created_at='2026-11-09T14:42:59.845038+00:00', parent_config={'configurable': {'thread_id': 'user-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1dc4bd-c187-6408-800e-6e4392b28fe9'}}, tasks=(), interrupts=())
